# Notebook 7: Differential Abundance Analysis (DESeq2)
To ensure a rigorous and granular evaluation, this notebook performs differential abundance testing using **DESeq2**. While global diversity metrics showed no macro-level shift between groups, this analysis investigates whether specific ASVs or bacterial taxa exhibit significant changes at the fine-grained level.

In [2]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
container_path = os.path.join(project_dir, "dada2.sif")

r_code = """
local_lib <- "/home/azureuser/Microbiome_project/R_libs"
.libPaths(c(local_lib, .libPaths()))

if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager", repos = "https://cloud.r-project.org", lib = local_lib)

cat("Installing DESeq2 locally...\n")
BiocManager::install("DESeq2", lib = local_lib, ask = FALSE, update = FALSE)
cat("DESeq2 installation completed successfully!\n")
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> local_lib <- "/home/azureuser/Microbiome_project/R_libs"
> .libPaths(c(local_lib, .libPaths()))
> 
> if (!requireNamespace("BiocManager", quietly = TRUE))
+     install.packages("BiocManager", repos = "https://cloud.r-project.org", lib = local_lib)
> 
> cat("Installing DESeq2 locally...
+ ")
Installing DESeq2 loc

In [4]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
container_path = os.path.join(project_dir, "dada2.sif")

r_code = """
local_lib <- "/home/azureuser/Microbiome_project/R_libs"
.libPaths(c(local_lib, .libPaths()))

library(phyloseq)
library(tidyverse)
library(DESeq2)

cat("Loading phyloseq object...\n")
ps <- readRDS("phyloseq_final.rds")

cat("Converting to DESeq2 object...\n")
diagdds <- phyloseq_to_deseq2(ps, ~ Supplementation)

# Use 'poscounts' estimator to handle zeros safely across rows
cat("Estimating size factors using poscounts method...\n")
diagdds <- estimateSizeFactors(diagdds, type = "poscounts")

cat("Running DESeq analysis...\n")
diagdds <- DESeq(diagdds, test="Wald", fitType="parametric")

# Extract results comparing L.reuteri vs Placebo
res <- results(diagdds, cooksCutoff = FALSE)
alpha <- 0.05
sig_tab <- res[which(res$padj < alpha), ]

if(nrow(sig_tab) > 0) {
    sig_tab <- cbind(as(sig_tab, "data.frame"), as(tax_table(ps)[rownames(sig_tab), ], "matrix"))
    cat(paste("\\nNumber of differentially abundant ASVs (padj < 0.05):", nrow(sig_tab), "\\n"))
    print(head(sig_tab[, c("baseMean", "log2FoldChange", "pvalue", "padj", "Genus", "Species")]))
    write.csv(sig_tab, "deseq2_significant_taxa.csv")
    cat("\\nSignificant taxa saved to deseq2_significant_taxa.csv!\n")
} else {
    cat("\\n=========================================================\n")
    cat("Result: No individual ASVs passed the FDR threshold (padj < 0.05).\n")
    cat("=========================================================\n")
}
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> local_lib <- "/home/azureuser/Microbiome_project/R_libs"
> .libPaths(c(local_lib, .libPaths()))
> 
> library(phyloseq)
> library(tidyverse)
> library(DESeq2)
> 
> cat("Loading phyloseq object...
+ ")
Loading phyloseq object...
> ps <- readRDS("phyloseq_final.rds")
> 
> cat("Converting to DESeq2 object...
+ ")
Con

In [5]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
container_path = os.path.join(project_dir, "dada2.sif")

r_code = """
local_lib <- "/home/azureuser/Microbiome_project/R_libs"
.libPaths(c(local_lib, .libPaths()))

library(phyloseq)
library(tidyverse)
library(DESeq2)
library(ggplot2)

cat("Loading phyloseq object...\n")
ps <- readRDS("phyloseq_final.rds")

cat("Preparing DESeq2 model for Volcano Plot...\n")
diagdds <- phyloseq_to_deseq2(ps, ~ Supplementation)
diagdds <- estimateSizeFactors(diagdds, type = "poscounts")
diagdds <- DESeq(diagdds, test="Wald", fitType="parametric")

# Extract results
res <- results(diagdds, cooksCutoff = FALSE)
res_df <- as.data.frame(res) %>% drop_na(pvalue)

# Add significance status column (even if none pass, it sets up the coloring correctly)
res_df$Significant <- ifelse(!is.na(res_df$padj) & res_df$padj < 0.05, "Significant (padj < 0.05)", "Not Significant")

# Generate Volcano Plot
cat("Generating Volcano Plot...\n")
p_volcano <- ggplot(res_df, aes(x = log2FoldChange, y = -log10(pvalue), color = Significant)) +
             geom_point(alpha = 0.6, size = 1.5) +
             scale_color_manual(values = c("Not Significant" = "grey60", "Significant (padj < 0.05)" = "red")) +
             theme_bw() +
             labs(title = "Volcano Plot: L. reuteri vs Placebo",
                  subtitle = "Differential Abundance Analysis (DESeq2)",
                  x = "Log2 Fold Change",
                  y = "-Log10 p-value") +
             geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "blue", alpha = 0.7)

# Save the plot
ggsave("volcano_plot.png", plot = p_volcano, width = 8, height = 6, dpi = 300)
cat("\nVolcano plot successfully saved to volcano_plot.png!\n")
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> local_lib <- "/home/azureuser/Microbiome_project/R_libs"
> .libPaths(c(local_lib, .libPaths()))
> 
> library(phyloseq)
> library(tidyverse)
> library(DESeq2)
> library(ggplot2)
> 
> cat("Loading phyloseq object...
+ ")
Loading phyloseq object...
> ps <- readRDS("phyloseq_final.rds")
> 
> cat("Preparing DESeq2 mod